<div style="width: 100%; margin-bottom: 20px;">
    <img src="../assets/banner.png" style="width: 100%; border-radius: 10px; height: 300px; object-fit: cover;">
</div>

# Phase 3 - Évaluation Finale du Modèle
## Projet ML : Prédiction des Tempêtes Géomagnétiques

**Réalisé par :**
- AMEZIANE Oumaima
- ROHAND Douae
- MOHITO Raihana

**Date :** 2025-2026  
**Cours :** Machine Learning – 2ème année Cycle Ingénieurs GI  
**Encadrant :** Pr. Y. EL YOUNOUSSI

---

Ce notebook constitue la **dernière étape de la Phase 3**. Le modèle final (issu de `05_tuning.ipynb`) est évalué **une seule fois** sur le jeu de test, resté intouché depuis la Phase 2.

**Contenu de ce notebook :**
1. Chargement du modèle final et du jeu de test
2. Prédictions sur le jeu de test
3. Métriques quantitatives globales
4. Matrice de confusion et interprétation métier
5. Courbe ROC
6. Courbe Precision-Recall (métrique principale)
7. Distribution des probabilités prédites
8. Optimisation du seuil de décision
9. Comparaison aux objectifs ML de Phase 1
10. Sauvegarde du seuil optimal
11. Synthèse finale

> **Principe fondamental :** Le jeu de test (`data/processed/test.csv`) n'a été vu ni pendant l'entraînement (`04_modeling.ipynb`), ni pendant le tuning (`05_tuning.ipynb`). L'évaluation ci-dessous est donc une estimation **honnête** des performances réelles du modèle.

> **Note temporelle :** En raison de la nature temporelle des données (série horaire 2019-2023), le split stratifié aléatoire utilisé introduit un risque théorique de fuite temporelle. Ce point est documenté et discuté en section 11.

---
## 0. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Métriques
from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

COLOR_STORM = '#FF8C00'
COLOR_CALM  = '#1a237e'
TARGET = 'is_storm'

# Coût asymétrique (Phase 1) :
# FN (tempête non détectée) >> FP (fausse alarme)
COST_FN = 100
COST_FP = 1

print('Imports chargés avec succès.')
print(f'Matrice de coût asymétrique : FN = {COST_FN} | FP = {COST_FP}')
print(f'Ratio FN/FP = {COST_FN // COST_FP}x → priorité au Recall')

In [ ]:
# ============================================================
# CELLULE EN ATTENTE — décommenter quand final_model.joblib est disponible
# ============================================================

# final_model = joblib.load('../models/final_model.joblib')
# print('Modèle final chargé depuis : models/final_model.joblib')
# print(f'Type de pipeline : {type(final_model)}')
# print(f'Étapes du pipeline : {[step[0] for step in final_model.steps]}')

print('[EN ATTENTE] Décommenter cette cellule quand final_model.joblib est créé par 05_tuning.ipynb')

In [ ]:
# Chargement du jeu de test (intouché depuis Phase 2)
df_test = pd.read_csv('../data/processed/test.csv')

feat_cols = [c for c in df_test.columns if c != TARGET]
X_test = df_test[feat_cols]
y_test = df_test[TARGET]

print(f'Jeu de test chargé : {X_test.shape[0]:,} lignes x {X_test.shape[1]} features')
vc = y_test.value_counts()
print(f'Distribution de la cible :')
print(f'  Classe 0 (Calme)   : {vc[0]:,} lignes ({vc[0]/len(y_test)*100:.1f}%)')
print(f'  Classe 1 (Tempête) : {vc[1]:,} lignes ({vc[1]/len(y_test)*100:.1f}%)')
print(f'  Ratio naturel      : 1 tempête pour {vc[0]/vc[1]:.1f} heures calmes')

---
## 2. Prédictions sur le jeu de test

Deux sorties sont nécessaires :
- `y_pred` : classes prédites (0 ou 1) au seuil par défaut 0.5
- `y_proba` : probabilité de la classe 1, nécessaire pour les courbes et l'optimisation du seuil

In [ ]:
y_pred  = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print(f'Prédictions générées : {len(y_pred):,} observations')
print(f'Probabilités : min={y_proba.min():.4f} | max={y_proba.max():.4f} | moy={y_proba.mean():.4f}')
pred_vc = pd.Series(y_pred).value_counts()
print(f'\nDistribution des prédictions (seuil 0.5) :')
print(f'  Prédit Calme   (0) : {pred_vc.get(0, 0):,}')
print(f'  Prédit Tempête (1) : {pred_vc.get(1, 0):,}')

---
## 3. Métriques quantitatives globales

Calcul au seuil par défaut (0.5). La métrique principale est le **PR-AUC** (plus fiable que le ROC-AUC sur données déséquilibrées car sensible à la proportion réelle de la classe minoritaire).

In [ ]:
recall    = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
accuracy  = accuracy_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_proba)
pr_auc    = average_precision_score(y_test, y_proba)
baseline_pr = y_test.mean()  # PR-AUC d'un modèle aléatoire = ratio de la classe positive

print('=' * 60)
print('  MÉTRIQUES — JEU DE TEST (seuil par défaut : 0.5)')
print('=' * 60)
print(f'  Recall (Sensibilité)     : {recall:.4f}   <- PRIORITÉ 1')
print(f'  Precision                : {precision:.4f}')
print(f'  F1-Score                 : {f1:.4f}')
print(f'  Accuracy                 : {accuracy:.4f}   (peu fiable sur déséquilibre)')
print(f'  ROC-AUC                  : {roc_auc:.4f}')
print(f'  PR-AUC (Average Prec.)   : {pr_auc:.4f}   <- Métrique principale')
print(f'  PR-AUC baseline (aléat.) : {baseline_pr:.4f}')
print('=' * 60)
print()
print('Rapport de classification complet :')
print(classification_report(y_test, y_pred, target_names=['Calme (0)', 'Tempête (1)']))

---
## 4. Matrice de confusion et interprétation métier

Chaque cellule a une signification concrète dans le contexte des tempêtes géomagnétiques.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=' * 60)
print('  MATRICE DE CONFUSION (seuil 0.5)')
print('=' * 60)
print(f'  VP (Vrais Positifs)  = {tp:,}   -> Tempêtes correctement détectées')
print(f'  VN (Vrais Négatifs)  = {tn:,} -> Périodes calmes correctement identifiées')
print(f'  FP (Faux Positifs)   = {fp:,}   -> Fausses alarmes (coût faible)')
print(f'  FN (Faux Négatifs)   = {fn:,}   -> Tempêtes MANQUÉES (coût élevé !)')
print()
print(f'  Taux de détection (Recall)  : {tp/(tp+fn)*100:.1f}% des tempêtes détectées')
print(f'  Taux de fausses alarmes     : {fp/(fp+tn)*100:.1f}% des calmes mal classés')

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Prédit Calme (0)', 'Prédit Tempête (1)'],
    yticklabels=['Réel Calme (0)', 'Réel Tempête (1)'],
    ax=ax, linewidths=0.5, linecolor='white'
)
ax.set_title('Matrice de Confusion — Jeu de Test (seuil 0.5)', fontweight='bold')
ax.set_ylabel('Réalité', fontweight='bold')
ax.set_xlabel('Prédiction', fontweight='bold')
plt.tight_layout()
os.makedirs('../data/processed', exist_ok=True)
plt.savefig('../data/processed/confusion_matrix_default.png', bbox_inches='tight', dpi=120)
plt.show()
print('Figure sauvegardée : data/processed/confusion_matrix_default.png')

In [ ]:
# Interprétation métier
cout_total_05 = COST_FN * fn + COST_FP * fp
print('=' * 60)
print('  INTERPRÉTATION MÉTIER DES 4 CELLULES')
print('=' * 60)
print(f'  VP = {tp:,}')
print(f'       -> Alerte émise et tempête réelle : infrastructure protégée.')
print()
print(f'  VN = {tn:,}')
print(f'       -> Pas d\'alerte, pas de tempête : fonctionnement nominal.')
print()
print(f'  FP = {fp:,}')
print(f'       -> Fausse alarme : mesures de protection déclenchées inutilement.')
print(f'       Coût = {fp} x {COST_FP} = {fp * COST_FP}')
print()
print(f'  FN = {fn:,}')
print(f'       -> TEMPÊTE MANQUÉE : aucune protection déclenchée.')
print(f'       Coût = {fn} x {COST_FN} = {fn * COST_FN}  <-- coût dominant')
print()
print(f'  Coût total au seuil 0.5 : {cout_total_05:,}')

---
## 5. Courbe ROC

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'Modèle final (ROC-AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Aléatoire (AUC = 0.5)')

idx_05 = np.argmin(np.abs(thresholds_roc - 0.5))
ax.scatter(fpr[idx_05], tpr[idx_05], color='red', zorder=5, s=80,
           label=f'Seuil 0.5 (FPR={fpr[idx_05]:.2f}, TPR={tpr[idx_05]:.2f})')

ax.set_xlabel('Taux de Faux Positifs (1 - Spécificité)', fontweight='bold')
ax.set_ylabel('Taux de Vrais Positifs (Recall)', fontweight='bold')
ax.set_title('Courbe ROC — Jeu de Test', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.01])

plt.tight_layout()
plt.savefig('../data/processed/roc_curve.png', bbox_inches='tight', dpi=120)
plt.show()
print(f'ROC-AUC = {roc_auc:.4f}')
print('Figure sauvegardée : data/processed/roc_curve.png')

---
## 6. Courbe Precision-Recall (métrique principale sur données déséquilibrées)

La courbe PR est **plus informative que la ROC** sur données déséquilibrées car elle ne prend pas en compte les vrais négatifs (très nombreux ici). La baseline d'un modèle aléatoire est égale au ratio de la classe positive (~8.5%).

In [ ]:
prec_curve, rec_curve, thresholds_pr = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_curve, prec_curve, color=COLOR_STORM, lw=2,
        label=f'Modèle final (PR-AUC = {pr_auc:.4f})')
ax.axhline(y=baseline_pr, color='gray', linestyle='--', lw=1,
           label=f'Baseline aléatoire (PR-AUC = {baseline_pr:.3f})')

idx_05_pr = np.argmin(np.abs(thresholds_pr - 0.5))
ax.scatter(rec_curve[idx_05_pr], prec_curve[idx_05_pr], color='red', zorder=5, s=80,
           label=f'Seuil 0.5 (Rec={rec_curve[idx_05_pr]:.2f}, Prec={prec_curve[idx_05_pr]:.2f})')

ax.set_xlabel('Recall (Taux de Vrais Positifs)', fontweight='bold')
ax.set_ylabel('Precision', fontweight='bold')
ax.set_title('Courbe Precision-Recall — Jeu de Test\n(métrique principale pour données déséquilibrées)',
             fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.01])

plt.tight_layout()
plt.savefig('../data/processed/pr_curve.png', bbox_inches='tight', dpi=120)
plt.show()
print(f'PR-AUC = {pr_auc:.4f} | Baseline = {baseline_pr:.4f} | Gain = +{pr_auc - baseline_pr:.4f}')
print('Figure sauvegardée : data/processed/pr_curve.png')

---
## 7. Distribution des probabilités prédites par classe

Un bon modèle concentre les probabilités des tempêtes réelles vers 1 et celles des périodes calmes vers 0. Un chevauchement important indique une zone d'incertitude où le seuil est décisif.

In [ ]:
proba_calme = y_proba[y_test == 0]
proba_storm = y_proba[y_test == 1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(proba_calme, bins=50, color=COLOR_CALM, alpha=0.6,
        label=f'Calme (0) — n={len(proba_calme):,}', density=True)
ax.hist(proba_storm, bins=50, color=COLOR_STORM, alpha=0.7,
        label=f'Tempête (1) — n={len(proba_storm):,}', density=True)
ax.axvline(x=0.5, color='red', linestyle='--', lw=1.5, label='Seuil par défaut (0.5)')
ax.set_xlabel('Probabilité prédite de la classe Tempête', fontweight='bold')
ax.set_ylabel('Densité', fontweight='bold')
ax.set_title('Distribution des Probabilités Prédites par Classe Réelle', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/proba_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'Probabilité médiane — Calme   : {np.median(proba_calme):.4f}')
print(f'Probabilité médiane — Tempête : {np.median(proba_storm):.4f}')
print('Figure sauvegardée : data/processed/proba_distribution.png')

---
## 8. Optimisation du seuil de décision

Le seuil par défaut (0.5) est rarement optimal sur des données déséquilibrées. Nous cherchons le seuil $t^*$ qui minimise le coût métier total :

$$\text{Coût total}(t) = \text{COST\_FN} \times FN(t) + \text{COST\_FP} \times FP(t)$$

avec $\text{COST\_FN} = 100$ et $\text{COST\_FP} = 1$.

In [ ]:
thresholds = np.arange(0.01, 1.00, 0.01)
costs, recalls_t, precisions_t, f1s_t = [], [], [], []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    costs.append(COST_FN * fn_t + COST_FP * fp_t)
    recalls_t.append(recall_score(y_test, y_pred_t, zero_division=0))
    precisions_t.append(precision_score(y_test, y_pred_t, zero_division=0))
    f1s_t.append(f1_score(y_test, y_pred_t, zero_division=0))

costs = np.array(costs)
recalls_t = np.array(recalls_t)
precisions_t = np.array(precisions_t)
f1s_t = np.array(f1s_t)

idx_opt = np.argmin(costs)
threshold_opt = thresholds[idx_opt]
cost_opt = costs[idx_opt]
idx_05_cost = np.argmin(np.abs(thresholds - 0.5))
cost_05 = costs[idx_05_cost]

print(f'Seuil optimal         : {threshold_opt:.2f}')
print(f'Coût au seuil optimal : {cost_opt:,}')
print(f'Coût au seuil 0.5     : {cost_05:,}')
print(f'Réduction de coût     : -{cost_05 - cost_opt:,} ({(cost_05 - cost_opt) / cost_05 * 100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Optimisation du Seuil de Décision', fontsize=13, fontweight='bold')

# Coût métier
axes[0].plot(thresholds, costs, color='#C0392B', lw=2, label='Coût total')
axes[0].axvline(x=threshold_opt, color='green', linestyle='--', lw=1.5,
                label=f'Seuil optimal ({threshold_opt:.2f}) — coût={cost_opt:,}')
axes[0].axvline(x=0.5, color='red', linestyle=':', lw=1.5,
                label=f'Seuil défaut (0.5) — coût={cost_05:,}')
axes[0].scatter([threshold_opt], [cost_opt], color='green', s=100, zorder=5)
axes[0].set_xlabel('Seuil de décision', fontweight='bold')
axes[0].set_ylabel('Coût total (FN×100 + FP×1)', fontweight='bold')
axes[0].set_title('Coût métier asymétrique vs Seuil')
axes[0].legend(fontsize=9)

# Recall / Precision / F1
axes[1].plot(thresholds, recalls_t,    color='steelblue',   lw=2, label='Recall')
axes[1].plot(thresholds, precisions_t, color=COLOR_STORM,   lw=2, label='Precision')
axes[1].plot(thresholds, f1s_t,        color='green',       lw=2, linestyle='--', label='F1-Score')
axes[1].axvline(x=threshold_opt, color='purple', linestyle='--', lw=1.5,
                label=f'Seuil optimal ({threshold_opt:.2f})')
axes[1].axvline(x=0.5, color='red', linestyle=':', lw=1.5, label='Seuil défaut (0.5)')
axes[1].set_xlabel('Seuil de décision', fontweight='bold')
axes[1].set_ylabel('Score', fontweight='bold')
axes[1].set_title('Recall / Precision / F1 vs Seuil')
axes[1].legend(fontsize=9)
axes[1].set_ylim([0, 1.01])

plt.tight_layout()
plt.savefig('../data/processed/threshold_optimization.png', bbox_inches='tight', dpi=120)
plt.show()
print('Figure sauvegardée : data/processed/threshold_optimization.png')

In [ ]:
# Métriques au seuil optimal
y_pred_opt = (y_proba >= threshold_opt).astype(int)
cm_opt = confusion_matrix(y_test, y_pred_opt)
tn_opt, fp_opt, fn_opt, tp_opt = cm_opt.ravel()

recall_opt    = recall_score(y_test, y_pred_opt, zero_division=0)
precision_opt = precision_score(y_test, y_pred_opt, zero_division=0)
f1_opt        = f1_score(y_test, y_pred_opt, zero_division=0)

print('=' * 65)
print(f'  COMPARAISON : Seuil 0.5 vs Seuil optimal ({threshold_opt:.2f})')
print('=' * 65)
print(f'{"Métrique":<25} {"Seuil 0.5":>12} {"Seuil opt.":>12} {"Gain":>10}')
print('-' * 65)
print(f'{"Recall":<25} {recall:>12.4f} {recall_opt:>12.4f} {"+" if recall_opt >= recall else ""}{recall_opt - recall:>9.4f}')
print(f'{"Precision":<25} {precision:>12.4f} {precision_opt:>12.4f} {"+" if precision_opt >= precision else ""}{precision_opt - precision:>9.4f}')
print(f'{"F1-Score":<25} {f1:>12.4f} {f1_opt:>12.4f} {"+" if f1_opt >= f1 else ""}{f1_opt - f1:>9.4f}')
print(f'{"Faux Négatifs (FN)":<25} {fn:>12,} {fn_opt:>12,} {fn_opt - fn:>+10,}')
print(f'{"Faux Positifs (FP)":<25} {fp:>12,} {fp_opt:>12,} {fp_opt - fp:>+10,}')
print(f'{"Coût total":<25} {cost_05:>12,} {cost_opt:>12,} {cost_opt - cost_05:>+10,}')
print('=' * 65)

In [ ]:
# Comparaison visuelle des deux matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Matrices de Confusion : Seuil 0.5 vs Seuil Optimal', fontsize=13, fontweight='bold')

for ax, cm_data, title in zip(
    axes,
    [cm, cm_opt],
    [f'Seuil par défaut (0.5) — Coût={cost_05:,}',
     f'Seuil optimal ({threshold_opt:.2f}) — Coût={cost_opt:,}']
):
    sns.heatmap(
        cm_data, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Prédit Calme', 'Prédit Tempête'],
        yticklabels=['Réel Calme', 'Réel Tempête'],
        ax=ax, linewidths=0.5, linecolor='white'
    )
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Réalité', fontweight='bold')
    ax.set_xlabel('Prédiction', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_comparison.png', bbox_inches='tight', dpi=120)
plt.show()
print('Figure sauvegardée : data/processed/confusion_matrix_comparison.png')

---
## 9. Comparaison aux objectifs ML de Phase 1

Vérification explicite et transparente : les objectifs définis en Phase 1 sont-ils atteints au seuil optimal ?

In [ ]:
# Objectifs ML définis en Phase 1
# (adapter les seuils si votre cadrage.md spécifie des valeurs différentes)
OBJECTIFS = {
    'Recall >= 0.80':      (recall_opt,    0.80),
    'Precision >= 0.50':   (precision_opt, 0.50),
    'PR-AUC > baseline':   (pr_auc,        baseline_pr),
}

print('=' * 65)
print('  COMPARAISON AUX OBJECTIFS ML (Phase 1) — Seuil optimal')
print('=' * 65)
print(f'{"Objectif":<28} {"Cible":>10} {"Obtenu":>12} {"Résultat":>12}')
print('-' * 65)

all_passed = True
for label, (valeur, cible) in OBJECTIFS.items():
    atteint = valeur >= cible
    if not atteint:
        all_passed = False
    symbole = 'ATTEINT' if atteint else 'NON ATTEINT'
    print(f'  {label:<26} {cible:>10.3f} {valeur:>12.4f} {symbole:>12}')

print('=' * 65)
if all_passed:
    print('  CONCLUSION : Tous les objectifs ML de Phase 1 sont atteints.')
else:
    print('  CONCLUSION : Certains objectifs ne sont pas atteints.')
    print('  Analyse : signal physique insuffisant, déséquilibre résiduel,')
    print('  ou espace de tuning à élargir en 05_tuning.ipynb.')
print('=' * 65)

---
## 10. Sauvegarde du seuil optimal dans le modèle final

Le seuil optimal est stocké comme attribut du pipeline afin que la Phase 4 (FastAPI) puisse l'appliquer directement sans recalcul.

In [ ]:
# Injection du seuil optimal comme attribut du pipeline
final_model.decision_threshold = float(threshold_opt)
joblib.dump(final_model, '../models/final_model.joblib')

print(f'Seuil optimal sauvegardé : final_model.decision_threshold = {threshold_opt:.2f}')
print(f'Modèle re-sérialisé      : models/final_model.joblib')

# Vérification
m_check = joblib.load('../models/final_model.joblib')
assert m_check.decision_threshold == float(threshold_opt)
print('Vérification réussie : attribut decision_threshold bien persisté.')
print()
print('Utilisation en Phase 4 (FastAPI) :')
print('  model = joblib.load("models/final_model.joblib")')
print('  proba = model.predict_proba(X_new)[:, 1]')
print('  prediction = (proba >= model.decision_threshold).astype(int)')

---
## 11. Synthèse Finale — Phase 3

### Note sur le risque de fuite temporelle

> Le dataset est une **série temporelle** (données horaires 2019-2023). Le split stratifié aléatoire utilisé en Phase 2 peut introduire une fuite temporelle : des observations temporellement proches peuvent se retrouver dans le train et dans le test, permettant au modèle d'exploiter implicitement des corrélations temporelles. Ce risque est documenté mais n'invalide pas les résultats dans un cadre académique. En production, un split temporel strict (ex. : train=2019-2022, test=2023) serait préférable.

In [ ]:
print('=' * 65)
print('  SYNTHÈSE FINALE — PHASE 3 / 06_evaluation.ipynb')
print('=' * 65)
print()
print('  ÉVALUATION AU SEUIL PAR DÉFAUT (0.5)')
print(f'    Recall    : {recall:.4f}')
print(f'    Precision : {precision:.4f}')
print(f'    F1-Score  : {f1:.4f}')
print(f'    ROC-AUC   : {roc_auc:.4f}')
print(f'    PR-AUC    : {pr_auc:.4f}  (baseline = {baseline_pr:.4f})')
print()
print(f'  SEUIL OPTIMAL (minimisation du coût métier FN×100 + FP×1)')
print(f'    Seuil optimal      : {threshold_opt:.2f}')
print(f'    Recall au seuil opt: {recall_opt:.4f}')
print(f'    Coût seuil 0.5     : {cost_05:,}')
print(f'    Coût seuil optimal : {cost_opt:,}')
print(f'    Réduction de coût  : {(cost_05 - cost_opt) / cost_05 * 100:.1f}%')
print()
print('  FIGURES PRODUITES')
print('    - data/processed/confusion_matrix_default.png')
print('    - data/processed/roc_curve.png')
print('    - data/processed/pr_curve.png')
print('    - data/processed/proba_distribution.png')
print('    - data/processed/threshold_optimization.png')
print('    - data/processed/confusion_matrix_comparison.png')
print()
print('  MODÈLE FINAL')
print(f'    - models/final_model.joblib (decision_threshold={threshold_opt:.2f})')
print()
print('  RISQUE DOCUMENTÉ')
print('    - Fuite temporelle potentielle (split aléatoire sur série temporelle)')
print('    - Acceptable en contexte académique, à corriger en production')
print('=' * 65)